In [ ]:
from pymongo import MongoClient
import socket
import json
import time

# Conexión con MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["northwind_mongo"]
collection = db["employees"]

# Conexión con Logstash (input tcp del employees_logstash.conf)
LOGSTASH_HOST = "127.0.0.1"
LOGSTASH_PORT = 6000

print(f"Conectando a Logstash en {LOGSTASH_HOST}:{LOGSTASH_PORT} ...")
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.connect((LOGSTASH_HOST, LOGSTASH_PORT))
print("Conectado. Empezando a vigilar la colección employees...\n")

ultimo_employee_id = 0  # arranca en 0: la primera pasada trae TODOS los empleados existentes

try:
    while True:
        nuevos = collection.find(
            {"employeeID": {"$gt": ultimo_employee_id}}
        ).sort("employeeID", 1)

        for doc in nuevos:
            # ObjectId no es serializable a JSON directo, y '_id' es un campo
            # reservado en Elasticsearch (no se puede mandar dentro del documento)
            doc["mongo_id"] = str(doc.pop("_id"))
            linea = json.dumps(doc) + "\n"  # json_lines: un documento por línea
            sock.sendall(linea.encode("utf-8"))

            print(
                f"Enviado a Logstash -> employeeID: {doc['employeeID']} | "
                f"{doc.get('firstName')} {doc.get('lastName')}"
            )

            ultimo_employee_id = doc["employeeID"]

        time.sleep(10)

except KeyboardInterrupt:
    print("\nPuente detenido.")

finally:
    sock.close()
    client.close()
    print("Conexiones cerradas.")


Conectando a Logstash en 127.0.0.1:6000 ...
Conectado. Empezando a vigilar la colección employees...

Enviado a Logstash -> employeeID: 1 | Nancy Davolio
Enviado a Logstash -> employeeID: 2 | Andrew Fuller
Enviado a Logstash -> employeeID: 3 | Janet Leverling
Enviado a Logstash -> employeeID: 4 | Margaret Peacock
Enviado a Logstash -> employeeID: 5 | Steven Buchanan
Enviado a Logstash -> employeeID: 6 | Michael Suyama
Enviado a Logstash -> employeeID: 7 | Robert King
Enviado a Logstash -> employeeID: 8 | Laura Callahan
Enviado a Logstash -> employeeID: 9 | Anne Dodsworth
Enviado a Logstash -> employeeID: 10 | Carlos Perez
Enviado a Logstash -> employeeID: 11 | Maria Torres
Enviado a Logstash -> employeeID: 12 | Diego Gomez
Enviado a Logstash -> employeeID: 13 | Jorge Diaz
Enviado a Logstash -> employeeID: 14 | Paula Diaz
Enviado a Logstash -> employeeID: 15 | Jorge Castro
Enviado a Logstash -> employeeID: 16 | Sofia Ramirez
Enviado a Logstash -> employeeID: 17 | Paula Castro
Enviado a 